In [ ]:
# /// script
# requires-python = ">=3.10"
# dependencies = [
#     "algebrax",
# ]
# [tool.uv.sources]
# algebrax = { path = ".." }
# ///

# Vibration Structural Analysis & Signal Processing

## Theory & Mathematical Foundation

1. **Permutation Symmetries (`algebrax.group.compose` & `ax.group.signature`)**:
   Permutation mappings represent spatial rotations and reflections of multi-rotor turbine systems.
   Composing permutations ($P_2 \circ P_1$) models sequential rotation,
   while `group.signature(P)` computes parity ($+1$ for even rotations, $-1$ for reflections).

2. **Structural Resonant Determinants (`algebrax.matrix.academic.determinant`)**:
   The determinant $\det(K - \omega^2 M)$ of the structural stiffness and mass matrix
   vanishes ($\det = 0$) at characteristic resonant frequencies.

3. **Analytic Signal Envelope (`algebrax.transforms.hilbert`)**:
   The Hilbert transform $\mathcal{H}\{x[n]\}$ forms an analytic signal $a[n] = x[n] + j \mathcal{H}\{x[n]\}$.
   The magnitude $|a[n]|$ represents the instantaneous amplitude envelope, isolating vibration spikes.

In [ ]:
import math

import algebrax as ax

## Step 1: Turbine Rotor Permutation Symmetries (Group Theory)

In [ ]:
rot_90 = {0: 1, 1: 2, 2: 3, 3: 0}
flip_h = {0: 1, 1: 0, 2: 3, 3: 2}

combined_motion = ax.group.compose(rot_90, flip_h)

print(f'90° Rotation Permutation (r): {rot_90} [Parity sgn: {ax.group.signature(rot_90):+d}]')
print(f'Horizontal Flip Permutation (s): {flip_h} [Parity sgn: {ax.group.signature(flip_h):+d}]')
print(f'Combined Motion (s o r):        {combined_motion} [Parity sgn: {ax.group.signature(combined_motion):+d}]')

## Step 2: Structural Stiffness Coupling Determinant (`ax.matrix.determinant`)

In [ ]:
stiffness_matrix = {
    0: {0: 4.0, 1: -2.0, 2: 0.0},
    1: {0: -2.0, 1: 5.0, 2: -3.0},
    2: {0: 0.0, 1: -3.0, 2: 4.0},
}

det_k = ax.matrix.academic.determinant(stiffness_matrix)
print('\nStiffness Matrix K:')
for r in sorted(stiffness_matrix.keys()):
    print(f'  Row {r}: {stiffness_matrix[r]}')

print(f'\nSystem Stiffness Determinant det(K): {det_k:.2f}')
if det_k > 0:
    print('Status: STABLE RIGID STRUCTURE (Non-singular, det(K) > 0)')
else:
    print('Status: WARNING - SINGULAR UNCONSTRAINED STRUCTURE (det(K) = 0)')

## Step 3: Instantaneous Vibration Envelope Extraction (`transforms.hilbert`)

In [ ]:
n_samples = 16
raw_vibration = {i: math.sin(2 * math.pi * i / 4) * (3.0 if 6 <= i <= 10 else 1.0) for i in range(n_samples)}

analytic_signal = ax.transforms.hilbert(raw_vibration, n=n_samples)

print('\nVibration Sensor Signal & Instantaneous Envelope:')
print('  Index | Raw Signal x[n] | Analytic Envelope |a[n]|')
print('  -----------------------------------------------')
for i in range(n_samples):
    raw_val = raw_vibration.get(i, 0.0)
    complex_val = analytic_signal.get(i, 0j)
    envelope_mag = abs(complex_val)
    tag = ' <== TRANSIENT BURST SPIKE' if envelope_mag > 2.0 else ''
    print(f'  {i:5d} | {raw_val:14.4f} | {envelope_mag:20.4f}{tag}')


def main() -> None:
    """Entry point for CLI execution."""
    print('==========================================================================')
    print('Recipe: Vibration Structural Analysis Finished Successfully!')
    print('==========================================================================')


if __name__ == '__main__':
    main()